# 랜덤 포레스트 실습

**Random Forest · RF**

서로 다르게 학습한 결정 트리의 예측을 평균하거나 다수결로 합치는 앙상블 모델.

소재 분야에서 이해하기: 수백 개 합금 데이터로 경도를 예측한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 앙상블 문서](https://scikit-learn.org/stable/modules/ensemble.html)

## 1. 트리 하나와 숲의 차이

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

for name, model in [('결정 트리 1개', DecisionTreeRegressor(random_state=0)),
                    ('트리 10개', RandomForestRegressor(n_estimators=10, random_state=0)),
                    ('트리 300개', RandomForestRegressor(n_estimators=300, random_state=0))]:
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
    print('%-12s MAE %.2f ± %.2f' % (name, -scores.mean(), scores.std()))

## 2. 트리 수에 따른 안정화

In [ ]:
counts = [1, 2, 5, 10, 25, 50, 100, 200, 400]
errors = [-cross_val_score(RandomForestRegressor(n_estimators=count, random_state=0), X, y,
                           cv=5, scoring='neg_mean_absolute_error').mean() for count in counts]
plt.plot(counts, errors, 'o-'); plt.xscale('log')
plt.xlabel('number of trees'); plt.ylabel('CV MAE (HV)'); plt.show()
print('트리를 늘리면 오차가 낮아지다가 평평해집니다. 과적합이 심해지지는 않지만 계산 시간이 늘어납니다.')

## 3. 숲은 구간을 벗어나면 평평해집니다

In [ ]:
model = RandomForestRegressor(n_estimators=300, random_state=0).fit(X, y)
sweep = np.column_stack([np.linspace(500, 1000, 200), np.full(200, 4.0), np.full(200, 2.0), np.zeros(200)])
plt.plot(sweep[:, 0], model.predict(sweep))
plt.axvspan(X[:, 0].min(), X[:, 0].max(), alpha=0.15, color='green')
plt.xlabel('sintering temperature (C)'); plt.ylabel('predicted hardness (HV)')
plt.title('green band = training range'); plt.show()
print('트리 기반 모델은 학습 범위 밖에서 상수로 평평해집니다. 외삽에 쓸 수 없습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#random-forest)을 여세요.